In [1]:
##Imports
import os
import sys
import time
from pathlib import Path
import numpy as np
import pandas as pd
import psycopg

sys.path.append(str(Path.cwd().parent))
from src.db import get_connection, execute_query, run_schema, nan_to_none

In [2]:
#Drop and import to tables, create engine and try query
engine = get_connection();

run_schema(engine)

execute_query("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'public'
    ORDER BY table_name
""", engine)

,table_name
0,nutrition
1,off_products
2,products
3,retailers


In [3]:
#Get OFF and products_matched data and resolve duplicates
PROCESSED = Path.cwd().parent / "data" / "processed"

off = pd.concat([
    pd.read_parquet(PROCESSED / "off_ch_clean.parquet").assign(scope="ch"),
    pd.read_parquet(PROCESSED / "off_aldi_ext_clean.parquet").assign(scope="de_at"),
    pd.read_parquet(PROCESSED / "off_openprices_ext_clean.parquet").assign(scope="openprices_ext")
], ignore_index=True)

#print(off.head())

# Deduplicate OFF by code, priority ch > de_at > openprices_ext
priority = {"ch": 0, "de_at": 1, "openprices_ext": 2}

off = (off.assign(_p=off.scope.map(priority))
          .sort_values("_p")
          .drop_duplicates(subset="code", keep="first")
          .drop(columns="_p"))

#print("after dedup:", len(off))

pm = pd.read_parquet(PROCESSED / "products_matched.parquet")

In [4]:
#Load retailers data

retailers = (pm[["retailer", "price_source"]]
            .drop_duplicates(subset="retailer")
            .rename(columns={"retailer": "retailer_id"})
            .sort_values("retailer_id")
                )

retailers.to_sql("retailers", engine, if_exists="append", index=False)
retailers 

,retailer_id,price_source
0,aldi,retailer_web
2022,coop,open_prices
2153,denner,open_prices
2168,lidl,open_prices
2201,migros,open_prices
1577,volg,retailer_web


In [6]:
#Load off data

off_cols = ["code", "scope", "product_name", "brands", "main_category",
            "pnns_group_1", "pnns_group_2", "nutriscore_grade", "nutriscore_score",
            "nova_group", "origin_status", "is_organic",
            "energy_kj", "fat", "saturated_fat", "carbohydrates",
            "sugars", "fiber", "proteins", "salt"]

off_out = off[off_cols].copy()
off_out["nutriscore_score"] = off_out.nutriscore_score.astype("Int64")   # INTEGER in schema
off_out["nova_group"]       = off_out.nova_group.astype("Int64")         # SMALLINT in schema

nan_to_none(off_out).to_sql("off_products", engine, if_exists="append", index=False, chunksize=2000, method="multi")


-45

In [7]:
#Load products data

pm = pm.reset_index(drop=True)
pm["product_id"] = pm.index + 1 

products = pd.DataFrame({
    "product_id":             pm.product_id,
    "retailer_id":            pm.retailer,                      
    "retailer_product_id":    pm.retailer_id.astype("string"),  
    "name":                   pm.name,
    "brand":                  pm.brand,
    "shop_category_main":     pm.category_main,
    "shop_category_sub":      pm.category_sub,
    "category_common":        pm.category_common,
    "category_source":        pm.category_common_source,
    "size_value":             pm.size_value,
    "size_unit":              pm.size_unit,
    "price_chf":              pm.price_chf,
    "price_per_100_chf":      pm.price_per_100_chf,
    "price_per_100_unit":     pm.price_per_100_unit,
    "price_source":           pm.price_source,
    "price_date":             pd.to_datetime(pm.price_date).dt.date,
    "is_organic":             pm.is_organic,
    "origin_status_retailer": pm.origin_status_retailer,
    "off_code":               pm.off_code.astype("string"),
    "match_method":           pm.match_method,
    "match_score":            pm.match_score,
})


nan_to_none(products).to_sql("products", engine, if_exists="append", index=False, chunksize=2000, method="multi")

-2

In [9]:
#Load products data

has = pm.nutrition_source.notna()

nutrition = pd.DataFrame({
     "product_id":           pm.loc[has, "product_id"],
    "nutrition_source":      pm.loc[has, "nutrition_source"],
    "energy_kj":             pm.loc[has, "energy_kj"],
    "fat":                   pm.loc[has, "fat"],
    "saturated_fat":         pm.loc[has, "saturated_fat"],
    "carbohydrates":         pm.loc[has, "carbohydrates"],
    "sugars":                pm.loc[has, "sugars"],
    "fiber":                 pm.loc[has, "fiber"],
    "proteins":              pm.loc[has, "proteins"],
    "salt":                  pm.loc[has, "salt"],
    "nutriscore_grade":      pm.loc[has, "nutriscore_grade"],
    "nutriscore_score":      pm.loc[has, "nutriscore_score"],
    "nutriscore_source":     pm.loc[has, "nutriscore_source"],
    "nutriscore_kind":       pm.loc[has, "nutriscore_kind"],
    "nutriscore_calc_grade": pm.loc[has, "nutriscore_calc_grade"],
    "nutriscore_calc_score": pm.loc[has, "nutriscore_calc_score"],
})


nan_to_none(nutrition).to_sql("nutrition", engine, if_exists="append", index=False, chunksize=2000, method="multi")

-1